# Asian FX Option Library — Tutorial

This notebook walks through the `asian_fx_option` library step by step.

Topics covered:
1. Installation & imports
2. Enums and data structures
3. Low-level helpers: `normalize_rate` and `weighted_average`
4. Cash settlement examples (fixed-strike Examples 1–3, floating-strike Example 4)
5. Physical settlement example (Example 5)
6. Custom weights and harmonic averaging
7. PUT options and out-of-the-money behaviour
8. Error handling

## 1. Installation

Install the library into your environment with either of:

```bash
# Using pip
pip install -e .

# Using uv (recommended)
uv pip install -e .
```

In [1]:
from asian_fx_option import (
    # High-level API
    asian_fx_option_payout,
    expected_cash_flow,
    expected_strike_physical,
    # Data structures
    AsianFXOptionSpec,
    # Enums
    OptionType,
    QuotationType,
    AveragingMethod,
    SettlementType,
    CurrencyType,
    # Low-level helpers
    normalize_rate,
    normalize_spread,
    weighted_average,
    # Exceptions
    AsianFXOptionError,
    InvalidWeightsError,
    MissingFinalRateError,
    MissingFixedStrikeError,
    MissingSettlementFixingError,
    ZeroAverageRateError,
    ZeroDivisionInAverageError,
)

print("Import successful.")

Import successful.


## 2. Enums

The library uses five enums to describe an option contract:

| Enum | Values | Meaning |
|---|---|---|
| `OptionType` | `CALL`, `PUT` | Payoff direction |
| `QuotationType` | `DIRECT`, `INDIRECT` | How the FX rate is quoted |
| `AveragingMethod` | `ARITHMETIC`, `HARMONIC` | Averaging method for fixings |
| `SettlementType` | `CASH`, `PHYSICAL` | Settlement type |
| `CurrencyType` | `BASE`, `QUOTE` | Which leg of the pair |

In [2]:
for enum_cls in (OptionType, QuotationType, AveragingMethod, SettlementType, CurrencyType):
    members = ", ".join(f"{m.name}={m.value!r}" for m in enum_cls)
    print(f"{enum_cls.__name__}: {members}")

OptionType: CALL='call', PUT='put'
QuotationType: DIRECT='direct', INDIRECT='indirect'
AveragingMethod: ARITHMETIC='arithmetic', HARMONIC='harmonic'
SettlementType: CASH='cash', PHYSICAL='physical'
CurrencyType: BASE='base', QUOTE='quote'


## 3. `AsianFXOptionSpec`

All parameters for a contract are bundled in a single `AsianFXOptionSpec` dataclass.

**Required fields:**

| Field | Type | Description |
|---|---|---|
| `option_type` | `OptionType` | CALL or PUT |
| `is_floating_strike` | `bool` | `True` → strike = multiplier × S_avg + spread; `False` → fixed strike |
| `strike_spread` | `float` | Additive spread in original quotation units |
| `quotation_type` | `QuotationType` | DIRECT or INDIRECT |
| `pair_scaling` | `int` | Scaling factor (e.g. 100 for JPY-style pips, 1 for standard) |
| `notional_currency` | `CurrencyType` | Currency in which notional is expressed |
| `notional_amount` | `float` | Notional size |
| `averaging_method` | `AveragingMethod` | ARITHMETIC or HARMONIC |
| `settlement_currency` | `CurrencyType` | Currency in which payout is expressed |
| `settlement_type` | `SettlementType` | CASH or PHYSICAL |

**Optional fields:**

| Field | Default | Description |
|---|---|---|
| `strike_multiplier` | `1.0` | Multiplier on S_avg for floating strike |
| `strike_fixed` | `None` | Fixed strike in original quotation units (required when `is_floating_strike=False`) |
| `final_rate_raw` | `None` | Spot/final rate in original quotation units — the payoff underlying for floating-strike cash settlement (required when `is_floating_strike=True` and `settlement_type=CASH`) |
| `fixings_weights` | `None` | Custom fixing weights; equal weights if `None` |
| `rounding_decimals` | `None` | Decimal places to round normalised strike |
| `settlement_fixing` | `None` | Settlement rate for base-currency cash settlement |

## 4. Low-level helpers

### `normalize_rate`

Converts a raw FX rate (original quotation, with scaling) to *quote per 1 base*:

- **DIRECT** (e.g. USD/SGD = 1.35 → 1 USD = 1.35 SGD): `rate_norm = raw / scale`  
- **INDIRECT** (e.g. SGD/USD = 0.74 → quote per base = 1/raw × scale): `rate_norm = scale / raw`

In [3]:
# DIRECT quote, scale=1: 1.35 USD/SGD stays 1.35
print("DIRECT  scale=1 :", normalize_rate(1.35, QuotationType.DIRECT, 1))

# DIRECT quote, scale=100: raw fixing of 85 (pips) -> 0.85
print("DIRECT  scale=100:", normalize_rate(85, QuotationType.DIRECT, 100))

# INDIRECT quote, scale=1: raw=1.35 -> norm = 1/1.35
print("INDIRECT scale=1 :", round(normalize_rate(1.35, QuotationType.INDIRECT, 1), 8))

DIRECT  scale=1 : 1.35
DIRECT  scale=100: 0.85
INDIRECT scale=1 : 0.74074074


### `weighted_average`

Computes a weighted arithmetic or harmonic average of a list of (already normalised) values.

- Pass `weights=None` for equal weights.  
- Weights are automatically re-normalised to sum to 1, so `[1, 3]` is equivalent to `[0.25, 0.75]`.

In [4]:
values = [1.30, 1.40]

# Equal weights
equal = weighted_average(values, weights=None, method=AveragingMethod.ARITHMETIC)
print("Equal weights     :", equal)  # 1.35

# Custom weights [1, 3] -> effective [0.25, 0.75]
custom = weighted_average(values, weights=[1.0, 3.0], method=AveragingMethod.ARITHMETIC)
print("Weights [1,3]     :", round(custom, 6))  # 1.375

# Harmonic mean of 1.0 and 2.0 = 2/(1/1 + 1/2) = 4/3
harm = weighted_average([1.0, 2.0], weights=None, method=AveragingMethod.HARMONIC)
print("Harmonic [1.0,2.0]:", round(harm, 6))  # 1.333333

Equal weights     : 1.35
Weights [1,3]     : 1.375
Harmonic [1.0,2.0]: 1.333333


## 5. Cash Settlement

For cash-settled options, `asian_fx_option_payout` (or `expected_cash_flow`) returns the payout amount in the settlement currency.

**Fixed-strike payoff logic (in normalised space):**

```
S_avg  = weighted_average(normalize(fixings))
K_norm = normalize(strike_fixed)

payoff_per_base = max(0, S_avg - K_norm)   # CALL
                = max(0, K_norm - S_avg)   # PUT

total_quote = base_notional × payoff_per_base
```

**Floating-strike payoff logic:**

The strike is derived from the average, but the *underlying* is the spot rate at expiry (`final_rate_raw`):

```
S_avg  = weighted_average(normalize(fixings))
K_norm = strike_multiplier × S_avg + spread_norm

F_norm (underlying) = normalize(final_rate_raw)

payoff_per_base = max(0, F_norm - K_norm)   # CALL
                = max(0, K_norm - F_norm)   # PUT
```

### Example 1 — Notional in BASE, settle in QUOTE

Scenario: 1-month USD/SGD Asian CALL.

- Pair: USD/SGD, direct quote, scale = 1  
- Fixings: `[1.35]`  
- Fixed strike: 1.34  
- Notional: 1,000,000 USD (base)  
- Settle: SGD (quote)

Expected payout = (1.35 − 1.34) × 1,000,000 = **10,000 SGD**

In [5]:
spec_ex1 = AsianFXOptionSpec(
    option_type=OptionType.CALL,
    is_floating_strike=False,
    strike_fixed=1.34,
    strike_spread=0.0,
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.BASE,
    notional_amount=1_000_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.CASH,
)

payout_ex1 = asian_fx_option_payout([1.35], spec_ex1)
print(f"Payout (SGD): {payout_ex1:,.2f}")  # 10,000.00

Payout (SGD): 10,000.00


### Example 2 — Notional in QUOTE, settle in QUOTE

Same trade but notional is expressed in SGD (quote currency).

- Notional: 1,350,000 SGD (quote) ↔ 1,000,000 USD at S_avg = 1.35  
- All other parameters unchanged

Expected payout = **10,000 SGD** (same result)

In [6]:
spec_ex2 = AsianFXOptionSpec(
    option_type=OptionType.CALL,
    is_floating_strike=False,
    strike_fixed=1.34,
    strike_spread=0.0,
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.QUOTE,
    notional_amount=1_350_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.CASH,
)

payout_ex2 = asian_fx_option_payout([1.35], spec_ex2)
print(f"Payout (SGD): {payout_ex2:,.2f}")  # 10,000.00

Payout (SGD): 10,000.00


### Example 3 — Notional in QUOTE, settle in BASE

Same trade but the payout is converted back to USD using a separate settlement fixing.

- Notional: 1,350,000 SGD (quote)  
- Settle: USD (base)  
- Settlement fixing: 1.35 (USD/SGD at settlement date)

Expected payout = 10,000 SGD ÷ 1.35 ≈ **7,407.41 USD**

In [7]:
spec_ex3 = AsianFXOptionSpec(
    option_type=OptionType.CALL,
    is_floating_strike=False,
    strike_fixed=1.34,
    strike_spread=0.0,
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.QUOTE,
    notional_amount=1_350_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.BASE,
    settlement_type=SettlementType.CASH,
    settlement_fixing=1.35,
)

payout_ex3 = asian_fx_option_payout([1.35], spec_ex3)
print(f"Payout (USD): {payout_ex3:,.6f}")  # ~7,407.407407

Payout (USD): 7,407.407407


### Example 4 — Floating-strike PUT (average strike option)

This is a **floating-strike** option: the strike is determined by the average of fixings. The payoff underlying is a separate final/spot rate at expiry (`final_rate_raw`).

- Fixings: `[1.35, 1.36, 1.34]` → S_avg = 1.35  
- Spread: +0.02 → K_norm = 1.35 + 0.02 = **1.37**  
- Final rate at expiry: 1.36 (the underlying against which the put is compared)  
- Notional: 1,000,000 USD (base)

Expected payout = max(0, 1.37 − 1.36) × 1,000,000 = **10,000 SGD**

In [8]:
spec_ex4_float = AsianFXOptionSpec(
    option_type=OptionType.PUT,
    is_floating_strike=True,
    strike_spread=0.02,          # +0.02 SGD/USD added to S_avg
    final_rate_raw=1.36,         # spot rate at expiry — the payoff underlying
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.BASE,
    notional_amount=1_000_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.CASH,
)

payout_ex4_float = asian_fx_option_payout([1.35, 1.36, 1.34], spec_ex4_float)
# S_avg = 1.35, K_norm = 1.37, underlying = 1.36
# payoff = max(0, 1.37 - 1.36) * 1M = 10,000
print(f"Payout (SGD): {payout_ex4_float:,.2f}")  # 10,000.00

Payout (SGD): 10,000.00


## 6. Physical Settlement

For physically-settled options, `asian_fx_option_payout` (or `expected_strike_physical`) returns the **strike rate** in the original quotation convention. Delivery occurs at this rate.

The strike is computed in normalised space and then converted back:

```
K_norm     = strike_multiplier × S_avg_norm + spread_norm
K_original = K_norm × scale          # DIRECT
           = scale  / K_norm         # INDIRECT
```

### Example 4 — Direct quote, scale = 100, additive spread

Scenario: USD/JPY (or similar) Asian option with 100-pip scaling.

- Fixings: `[0.85, 0.86, 0.84]` (raw values; normalised = divide by 100)  
- Floating strike with additive spread = 0.02 (raw units)  
- pair_scaling = 100

Calculation:
```
S_avg_norm = mean([0.0085, 0.0086, 0.0084]) = 0.0085
K_norm     = 0.0085 + (0.02 / 100)          = 0.0085 + 0.0002 = 0.0087
K_original = 0.0087 × 100                   = 0.87
```

In [9]:
spec_ex4 = AsianFXOptionSpec(
    option_type=OptionType.CALL,
    is_floating_strike=True,
    strike_spread=0.02,       # in raw (scaled) units
    quotation_type=QuotationType.DIRECT,
    pair_scaling=100,
    notional_currency=CurrencyType.BASE,
    notional_amount=1_000_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.PHYSICAL,
)

strike_ex4 = asian_fx_option_payout([0.85, 0.86, 0.84], spec_ex4)
print(f"Expected strike: {round(strike_ex4, 6)}")  # 0.87

Expected strike: 0.87


## 7. Custom Weights and Harmonic Averaging

### Custom Weights

Pass `fixings_weights` to give later fixings (e.g. month-end) more weight. Weights are normalised automatically.

In [10]:
# Two fixings: early-month 1.30, late-month 1.40
# Weight 1 : 3 -> effective 25% / 75%
spec_cw = AsianFXOptionSpec(
    option_type=OptionType.CALL,
    is_floating_strike=True,
    strike_spread=0.0,
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.BASE,
    notional_amount=1_000_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.PHYSICAL,
    fixings_weights=[1.0, 3.0],
)

strike_cw = asian_fx_option_payout([1.30, 1.40], spec_cw)
print(f"Equal-weight strike : {weighted_average([1.30, 1.40], None, AveragingMethod.ARITHMETIC)}")  # 1.35
print(f"Custom-weight strike: {round(strike_cw, 6)}")  # 1.375 = 0.25*1.30 + 0.75*1.40

Equal-weight strike : 1.35
Custom-weight strike: 1.375


### Harmonic Averaging

Harmonic averaging is commonly used for rate products where the average of rates should reflect the average cost of converting currencies. The harmonic mean is always ≤ the arithmetic mean.

```
H = n / (1/x₁ + 1/x₂ + … + 1/xₙ)
```

In [11]:
fixings = [1.0, 2.0]

spec_harm = AsianFXOptionSpec(
    option_type=OptionType.CALL,
    is_floating_strike=True,
    strike_spread=0.0,
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.BASE,
    notional_amount=1_000_000.0,
    averaging_method=AveragingMethod.HARMONIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.PHYSICAL,
)

arith = weighted_average(fixings, None, AveragingMethod.ARITHMETIC)
harm  = weighted_average(fixings, None, AveragingMethod.HARMONIC)

print(f"Arithmetic mean: {arith}")          # 1.5
print(f"Harmonic mean  : {round(harm, 6)}")  # 1.333333 = 4/3
print(f"Harmonic ≤ Arithmetic: {harm <= arith}")

strike_harm = asian_fx_option_payout(fixings, spec_harm)
print(f"\nPhysical strike (harmonic): {round(strike_harm, 6)}")  # 1.333333

Arithmetic mean: 1.5
Harmonic mean  : 1.333333
Harmonic ≤ Arithmetic: True

Physical strike (harmonic): 1.333333


## 8. PUT Options and Out-of-the-Money Behaviour

A CALL pays off when S_avg > K; a PUT pays off when S_avg < K. Both return **zero** when out of the money.

In [12]:
base_spec = dict(
    is_floating_strike=False,
    strike_fixed=1.36,
    strike_spread=0.0,
    quotation_type=QuotationType.DIRECT,
    pair_scaling=1,
    notional_currency=CurrencyType.BASE,
    notional_amount=1_000_000.0,
    averaging_method=AveragingMethod.ARITHMETIC,
    settlement_currency=CurrencyType.QUOTE,
    settlement_type=SettlementType.CASH,
)

spec_put_itm = AsianFXOptionSpec(option_type=OptionType.PUT, **base_spec)
spec_call_otm = AsianFXOptionSpec(option_type=OptionType.CALL, **base_spec)

fixings = [1.35]  # S_avg = 1.35, K = 1.36

print(f"PUT  ITM payout: {asian_fx_option_payout(fixings, spec_put_itm):,.2f} SGD")   # 10,000.00
print(f"CALL OTM payout: {asian_fx_option_payout(fixings, spec_call_otm):,.2f} SGD")  # 0.00

PUT  ITM payout: 10,000.00 SGD
CALL OTM payout: 0.00 SGD


## 9. Error Handling

The library raises specific exceptions for invalid inputs. All exceptions inherit from `AsianFXOptionError`.

| Exception | When raised |
|---|---|
| `InvalidWeightsError` | Weights are negative, wrong length, or sum to zero |
| `MissingFixedStrikeError` | `is_floating_strike=False` but `strike_fixed` not provided |
| `MissingFinalRateError` | `is_floating_strike=True` and `settlement_type=CASH` but `final_rate_raw` not provided |
| `MissingSettlementFixingError` | `settlement_currency=BASE` but `settlement_fixing` not provided |
| `ZeroDivisionInAverageError` | Harmonic average encounters a zero fixing |
| `ZeroAverageRateError` | S_avg is zero when converting quote notional to base |

In [13]:
# MissingFinalRateError: floating-strike cash settlement without final_rate_raw
try:
    spec_no_final = AsianFXOptionSpec(
        option_type=OptionType.CALL,
        is_floating_strike=True,   # floating strike requires final_rate_raw for cash settlement
        strike_spread=0.0,
        # final_rate_raw omitted
        quotation_type=QuotationType.DIRECT,
        pair_scaling=1,
        notional_currency=CurrencyType.BASE,
        notional_amount=1_000_000.0,
        averaging_method=AveragingMethod.ARITHMETIC,
        settlement_currency=CurrencyType.QUOTE,
        settlement_type=SettlementType.CASH,
    )
    asian_fx_option_payout([1.35], spec_no_final)
except MissingFinalRateError as e:
    print(f"MissingFinalRateError: {e}")

MissingFinalRateError: final_rate_raw must be provided when is_floating_strike=True


In [14]:
# InvalidWeightsError: 3 weights provided for 2 fixings
try:
    spec_bad = AsianFXOptionSpec(
        option_type=OptionType.CALL,
        is_floating_strike=True,
        strike_spread=0.0,
        quotation_type=QuotationType.DIRECT,
        pair_scaling=1,
        notional_currency=CurrencyType.BASE,
        notional_amount=1_000_000.0,
        averaging_method=AveragingMethod.ARITHMETIC,
        settlement_currency=CurrencyType.QUOTE,
        settlement_type=SettlementType.CASH,
        fixings_weights=[0.5, 0.3, 0.2],  # 3 weights but only 2 fixings below
    )
    asian_fx_option_payout([1.35, 1.36], spec_bad)
except InvalidWeightsError as e:
    print(f"InvalidWeightsError: {e}")

InvalidWeightsError: Number of weights (3) must equal number of values (2)


In [15]:
# MissingSettlementFixingError: settle in BASE but settlement_fixing not provided
try:
    spec_no_settle = AsianFXOptionSpec(
        option_type=OptionType.CALL,
        is_floating_strike=False,
        strike_fixed=1.34,
        strike_spread=0.0,
        quotation_type=QuotationType.DIRECT,
        pair_scaling=1,
        notional_currency=CurrencyType.BASE,
        notional_amount=1_000_000.0,
        averaging_method=AveragingMethod.ARITHMETIC,
        settlement_currency=CurrencyType.BASE,  # needs settlement_fixing
        settlement_type=SettlementType.CASH,
        # settlement_fixing omitted
    )
    asian_fx_option_payout([1.35], spec_no_settle)
except MissingSettlementFixingError as e:
    print(f"MissingSettlementFixingError: {e}")

MissingSettlementFixingError: settlement_fixing must be provided when settlement_currency == BASE


In [16]:
# MissingFixedStrikeError: is_floating_strike=False but strike_fixed not given
try:
    spec_no_strike = AsianFXOptionSpec(
        option_type=OptionType.CALL,
        is_floating_strike=False,  # fixed strike mode
        strike_spread=0.0,
        strike_fixed=None,          # but no value supplied
        quotation_type=QuotationType.DIRECT,
        pair_scaling=1,
        notional_currency=CurrencyType.BASE,
        notional_amount=1_000_000.0,
        averaging_method=AveragingMethod.ARITHMETIC,
        settlement_currency=CurrencyType.QUOTE,
        settlement_type=SettlementType.CASH,
    )
    asian_fx_option_payout([1.35], spec_no_strike)
except MissingFixedStrikeError as e:
    print(f"MissingFixedStrikeError: {e}")

MissingFixedStrikeError: strike_fixed must be provided when is_floating_strike=False


In [17]:
# ZeroDivisionInAverageError: harmonic average with a zero fixing
try:
    spec_harm_zero = AsianFXOptionSpec(
        option_type=OptionType.CALL,
        is_floating_strike=True,
        strike_spread=0.0,
        final_rate_raw=1.35,
        quotation_type=QuotationType.DIRECT,
        pair_scaling=1,
        notional_currency=CurrencyType.BASE,
        notional_amount=1_000_000.0,
        averaging_method=AveragingMethod.HARMONIC,
        settlement_currency=CurrencyType.QUOTE,
        settlement_type=SettlementType.CASH,
    )
    asian_fx_option_payout([1.35, 0.0], spec_harm_zero)  # zero fixing causes division by zero
except ZeroDivisionInAverageError as e:
    print(f"ZeroDivisionInAverageError: {e}")

ZeroDivisionInAverageError: Harmonic average requires all fixings to be non-zero


In [18]:
# ZeroAverageRateError: quote notional with S_avg = 0
try:
    spec_zero_avg = AsianFXOptionSpec(
        option_type=OptionType.CALL,
        is_floating_strike=False,
        strike_fixed=0.0,
        strike_spread=0.0,
        quotation_type=QuotationType.DIRECT,
        pair_scaling=1,
        notional_currency=CurrencyType.QUOTE,  # needs S_avg to convert notional
        notional_amount=1_000_000.0,
        averaging_method=AveragingMethod.ARITHMETIC,
        settlement_currency=CurrencyType.QUOTE,
        settlement_type=SettlementType.CASH,
    )
    asian_fx_option_payout([0.0], spec_zero_avg)  # S_avg = 0 -> division by zero
except ZeroAverageRateError as e:
    print(f"ZeroAverageRateError: {e}")

ZeroAverageRateError: Cannot convert quote notional to base when S_avg is zero


## Summary

| Function | Returns | Use for |
|---|---|---|
| `asian_fx_option_payout(fixings, spec)` | payout or strike | Top-level API, dispatches on `settlement_type` |
| `expected_cash_flow(fixings, spec)` | cash payout in settlement ccy | Cash-settled trades |
| `expected_strike_physical(fixings, spec)` | strike in original convention | Physically-settled trades |

**Quick checklist when building a `spec`:**

- Set `is_floating_strike=True/False` and provide `strike_fixed` when `False`.
- Match `pair_scaling` to how your fixings are quoted (1 for standard, 100 for JPY pips, etc.).
- If `settlement_currency=BASE`, always supply `settlement_fixing`.
- Supply `fixings_weights` when fixings are not equally weighted.
- Use `AveragingMethod.HARMONIC` when the product term sheet specifies harmonic averaging.